# Play Policy Bot

Click a source square, then a target square. The bot uses explicit legal move masking and does not use search.

This notebook defaults to the latest completed checkpoint and CPU inference so it can be used without competing for CUDA memory.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [ ]:
import chess

from mcchess.bots import PolicyOnlyBot
from mcchess.bots.notebook import create_notebook_game

In [ ]:
run_dir = project_root / "runs" / "lichess_2026_05_2000plus_epoch20_cached_batchmetrics"
checkpoint_candidates = [
    run_dir / "checkpoint_latest.pt",
    run_dir / "checkpoint.pt",
    run_dir / "checkpoint_epoch_020.pt",
]
checkpoint_path = next((path for path in checkpoint_candidates if path.exists()), None)
if checkpoint_path is None:
    raise FileNotFoundError("No playable checkpoint found in the run directory.")

# Keep this on CPU while the main training run is using CUDA. Change to "auto"
# or "cuda" after training finishes if you want GPU inference.
inference_device = "cpu"

{"checkpoint": str(checkpoint_path), "device": inference_device}

In [ ]:
bot = PolicyOnlyBot.from_checkpoint(checkpoint_path, device=inference_device)
metadata = bot.checkpoint.metadata
{
    "checkpoint": str(metadata.path),
    "epoch": metadata.epoch,
    "completed_at": metadata.completed_at,
    "device": str(bot.device),
}

In [ ]:
from IPython.display import display

game = create_notebook_game(bot, human_color=chess.WHITE)
display(game)